In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

In [10]:
from transformers import AutoModel

# Load the model
model = AutoModel.from_pretrained('keepitreal/vietnamese-sbert')

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('keepitreal/vietnamese-sbert')

# Print the model architecture
print(model)

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [3]:
# Print the model configuration
print(model.config)

RobertaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "keepitreal/vietnamese-sbert",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 258,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "tokenizer_class": "PhobertTokenizer",
  "torch_dtype": "float32",
  "transformers_version": "4.46.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 64001
}



In [4]:
# Input sentences
sentence1 = "quả cam ngon ."
sentence2 = "quả táo dở ."

In [5]:
# Tokenize each sentence separately
def get_encoding(sentence):
    encoding = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True, max_length=512)
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    token_type_ids = encoding.get('token_type_ids', torch.zeros_like(input_ids))  # MiniLM may not use token_type_ids
    return input_ids, attention_mask, token_type_ids

In [6]:
# Compute embeddings
def get_embedding(input_ids, attention_mask):
    output = model(input_ids=input_ids, attention_mask=attention_mask)
    embeddings = output['last_hidden_state']
    attention_mask = attention_mask.unsqueeze(-1)  # Expand mask for broadcasting
    
    # Mean pooling (ignoring padding tokens)
    sentence_embedding = torch.sum(embeddings * attention_mask, dim=1) / attention_mask.sum(dim=1)
    
    # Normalize embeddings to unit vectors
    return F.normalize(sentence_embedding, p=2, dim=1)

In [11]:
# Get encodings for both sentences
input_ids1, attention_mask1, token_type_ids1 = get_encoding(sentence1)
input_ids2, attention_mask2, token_type_ids2 = get_encoding(sentence2)

In [12]:
# Convert tokens to human-readable format
tokens1 = tokenizer.convert_ids_to_tokens(input_ids1.squeeze().tolist())
tokens2 = tokenizer.convert_ids_to_tokens(input_ids2.squeeze().tolist())

In [13]:
# Display tokenized results
print(f"\nInput Sentence 1: {sentence1}")
print(f"Tokens: {tokens1}")
print(f"Token IDs: {input_ids1.squeeze().tolist()}")
print(f"Attention Mask: {attention_mask1.squeeze().tolist()}")
print(f"Token Type IDs: {token_type_ids1.squeeze().tolist()}")

print(f"\nInput Sentence 2: {sentence2}")
print(f"Tokens: {tokens2}")
print(f"Token IDs: {input_ids2.squeeze().tolist()}")
print(f"Attention Mask: {attention_mask2.squeeze().tolist()}")
print(f"Token Type IDs: {token_type_ids2.squeeze().tolist()}")


Input Sentence 1: quả cam ngon .
Tokens: ['<s>', 'quả', 'cam', 'ngon', '.', '</s>']
Token IDs: [0, 645, 2947, 1325, 5, 2]
Attention Mask: [1, 1, 1, 1, 1, 1]
Token Type IDs: [0, 0, 0, 0, 0, 0]

Input Sentence 2: quả táo dở .
Tokens: ['<s>', 'quả', 'táo', 'dở', '.', '</s>']
Token IDs: [0, 645, 4451, 6305, 5, 2]
Attention Mask: [1, 1, 1, 1, 1, 1]
Token Type IDs: [0, 0, 0, 0, 0, 0]


In [14]:
# Check max position embeddings
max_position_embeddings = model.config.max_position_embeddings  # Typically 512 for MiniLM
print(f"\nMax Position Embeddings: {max_position_embeddings}")


Max Position Embeddings: 258


In [15]:
# Ensure input length doesn't exceed max position embeddings
input_length1 = input_ids1.shape[1]
input_length2 = input_ids2.shape[1]
print(f"Input Sequence Length (Sentence 1): {input_length1}")
print(f"Input Sequence Length (Sentence 2): {input_length2}")

Input Sequence Length (Sentence 1): 6
Input Sequence Length (Sentence 2): 6


In [17]:
if input_length1 > max_position_embeddings:
    print(f"Warning: Sentence 1 exceeds max position embeddings. Truncating sequence.")
    input_ids1 = input_ids1[:, :max_position_embeddings]
    attention_mask1 = attention_mask1[:, :max_position_embeddings]

if input_length2 > max_position_embeddings:
    print(f"Warning: Sentence 2 exceeds max position embeddings. Truncating sequence.")
    input_ids2 = input_ids2[:, :max_position_embeddings]
    attention_mask2 = attention_mask2[:, :max_position_embeddings]

In [18]:
# Compute embeddings
embedding1 = get_embedding(input_ids1, attention_mask1)
embedding2 = get_embedding(input_ids2, attention_mask2)

# Compute cosine similarity
cosine_sim = F.cosine_similarity(embedding1, embedding2, dim=1)

# Print similarity score
print(f"\nCosine Similarity: {cosine_sim.item():.4f}")


Cosine Similarity: 0.6267


In [19]:
# Inspect individual embeddings
word_embeddings = model.embeddings.word_embeddings
position_embeddings = model.embeddings.position_embeddings
token_type_embeddings = model.embeddings.token_type_embeddings if hasattr(model.embeddings, 'token_type_embeddings') else None

In [20]:
print(f"\n--- Word Embeddings ---")
print(f"Word Embeddings Shape: {word_embeddings.weight.shape}")
print(f"Word Embeddings Example (for first sentence): {word_embeddings(input_ids1).squeeze(0).detach().cpu().numpy()[:5]}")  # Show first 5 tokens for brevity


--- Word Embeddings ---
Word Embeddings Shape: torch.Size([64001, 768])
Word Embeddings Example (for first sentence): [[ 0.04776255 -0.03108022  0.03463487 ...  0.01482091 -0.00282975
  -0.06773947]
 [ 0.00539031 -0.02485302  0.025632   ...  0.03325172 -0.06940553
  -0.04624826]
 [-0.04885882  0.03953361  0.08419622 ... -0.04867305 -0.08255015
  -0.07453709]
 [ 0.08800032  0.02175597  0.05956177 ...  0.01547021 -0.02286974
  -0.0438781 ]
 [-0.00733265 -0.02799174 -0.03265276 ...  0.02454297 -0.00775733
  -0.00114423]]


In [21]:
print(f"\n--- Position Embeddings ---")
print(f"Position Embeddings Shape: {position_embeddings.weight.shape}")
position_indices = torch.arange(input_length1).unsqueeze(0)
position_embedding_example = position_embeddings(position_indices)
print(f"Position Embeddings Example: {position_embedding_example.squeeze(0).detach().cpu().numpy()[:5]}")  # Show first 5 tokens


--- Position Embeddings ---
Position Embeddings Shape: torch.Size([258, 768])
Position Embeddings Example: [[ 0.01503538 -0.00270233 -0.00691505 ... -0.00213593  0.007655
   0.01075593]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [-0.03959696 -0.01856966 -0.09306313 ... -0.04326111  0.05854748
   0.0380499 ]
 [-0.02214354 -0.00969479 -0.03618965 ...  0.02331159  0.03571284
   0.0169634 ]
 [-0.01884145  0.03388012 -0.01707573 ...  0.01579634  0.06969938
   0.01350085]]


In [22]:
if token_type_embeddings is not None:
    print(f"\n--- Token Type Embeddings ---")
    print(f"Token Type Embeddings Shape: {token_type_embeddings.weight.shape}")
    print(f"Token Type Embeddings Example (for sentence 1): {token_type_embeddings(token_type_ids1).squeeze(0).detach().cpu().numpy()[:5]}")

# Display encoder details
encoder_output1 = model(input_ids=input_ids1, attention_mask=attention_mask1)['last_hidden_state']
encoder_output2 = model(input_ids=input_ids2, attention_mask=attention_mask2)['last_hidden_state']

print(f"\n--- Encoder Output ---")
print(f"Encoder Output Shape (Sentence 1): {encoder_output1.shape}")
print(f"Encoder Output Shape (Sentence 2): {encoder_output2.shape}")


--- Token Type Embeddings ---
Token Type Embeddings Shape: torch.Size([1, 768])
Token Type Embeddings Example (for sentence 1): [[-3.7262350e-04 -7.5953976e-05 -2.1880587e-04 ... -1.3324158e-05
   1.2803503e-04  4.1191559e-04]
 [-3.7262350e-04 -7.5953976e-05 -2.1880587e-04 ... -1.3324158e-05
   1.2803503e-04  4.1191559e-04]
 [-3.7262350e-04 -7.5953976e-05 -2.1880587e-04 ... -1.3324158e-05
   1.2803503e-04  4.1191559e-04]
 [-3.7262350e-04 -7.5953976e-05 -2.1880587e-04 ... -1.3324158e-05
   1.2803503e-04  4.1191559e-04]
 [-3.7262350e-04 -7.5953976e-05 -2.1880587e-04 ... -1.3324158e-05
   1.2803503e-04  4.1191559e-04]]

--- Encoder Output ---
Encoder Output Shape (Sentence 1): torch.Size([1, 6, 768])
Encoder Output Shape (Sentence 2): torch.Size([1, 6, 768])


In [23]:
# Inspect individual encoder layers
print(f"\n--- Encoder Layer Details ---")
for i, layer in enumerate(model.encoder.layer):
    print(f"\nLayer {i+1} Details:")
    
    # Attention details
    self_attention = layer.attention.self
    print(f"Self-Attention (Layer {i+1})")
    print(f"Query Projection Weights: {self_attention.query.weight.shape}")
    print(f"Key Projection Weights: {self_attention.key.weight.shape}")
    print(f"Value Projection Weights: {self_attention.value.weight.shape}")
    print(f"Output Projection Weights: {layer.attention.output.dense.weight.shape}")

    # Feedforward details
    intermediate = layer.intermediate
    output_layer = layer.output
    print(f"Feedforward Layer (Layer {i+1})")
    print(f"Intermediate Layer Weights: {intermediate.dense.weight.shape}")
    print(f"Output Layer Weights: {output_layer.dense.weight.shape}")


--- Encoder Layer Details ---

Layer 1 Details:
Self-Attention (Layer 1)
Query Projection Weights: torch.Size([768, 768])
Key Projection Weights: torch.Size([768, 768])
Value Projection Weights: torch.Size([768, 768])
Output Projection Weights: torch.Size([768, 768])
Feedforward Layer (Layer 1)
Intermediate Layer Weights: torch.Size([3072, 768])
Output Layer Weights: torch.Size([768, 3072])

Layer 2 Details:
Self-Attention (Layer 2)
Query Projection Weights: torch.Size([768, 768])
Key Projection Weights: torch.Size([768, 768])
Value Projection Weights: torch.Size([768, 768])
Output Projection Weights: torch.Size([768, 768])
Feedforward Layer (Layer 2)
Intermediate Layer Weights: torch.Size([3072, 768])
Output Layer Weights: torch.Size([768, 3072])

Layer 3 Details:
Self-Attention (Layer 3)
Query Projection Weights: torch.Size([768, 768])
Key Projection Weights: torch.Size([768, 768])
Value Projection Weights: torch.Size([768, 768])
Output Projection Weights: torch.Size([768, 768])
Fee

In [24]:
# Pooler Layer (ensure it's available and print the output)
if hasattr(model, 'pooler'):
    pooler_output1 = model(input_ids=input_ids1, attention_mask=attention_mask1)['pooler_output']
    pooler_output2 = model(input_ids=input_ids2, attention_mask=attention_mask2)['pooler_output']

    print(f"\n--- Pooler Layer ---")
    print(f"Pooler Output Shape (Sentence 1): {pooler_output1.shape}")
    print(f"Pooler Output Shape (Sentence 2): {pooler_output2.shape}")
    print(f"Pooler Output Example (Sentence 1): {pooler_output1}")
    print(f"Pooler Output Example (Sentence 2): {pooler_output2}")
    print(f"Pooler Dense Layer: {model.pooler.dense}")
    print(f"Pooler Activation: {model.pooler.activation}")
else:
    print("\nNo Pooler Layer found in the model.")


--- Pooler Layer ---
Pooler Output Shape (Sentence 1): torch.Size([1, 768])
Pooler Output Shape (Sentence 2): torch.Size([1, 768])
Pooler Output Example (Sentence 1): tensor([[ 1.9275e-01,  2.4446e-01,  5.2727e-02, -7.1383e-02, -6.1102e-02,
          8.9643e-02, -1.6069e-01,  1.4761e-02,  6.7082e-04, -2.1225e-01,
         -2.4119e-01,  1.2455e-01, -8.2246e-02, -2.8441e-01,  3.6447e-01,
          5.2043e-03,  1.5969e-01,  3.3191e-01, -3.3429e-01, -1.3879e-01,
          1.7183e-01, -1.5754e-01,  7.7089e-03,  7.8906e-03, -2.4447e-01,
         -5.4209e-01, -1.6465e-01,  1.0134e-01,  1.5213e-02, -8.1256e-02,
         -3.4041e-03,  1.6206e-01,  6.2483e-03, -2.1828e-01,  2.1302e-01,
         -2.3745e-01, -5.0374e-01, -8.6605e-02, -1.9065e-01,  4.1517e-02,
          1.8060e-01,  4.8318e-02, -3.0748e-01, -3.3283e-02,  2.0690e-01,
          1.2775e-01,  2.4669e-01,  2.9537e-01, -1.0540e-01,  4.0983e-02,
          3.5817e-01,  1.3801e-01,  8.1071e-02,  1.4341e-01, -2.4478e-01,
         -4.4738e-